In [ ]:
#@title Step 3 — Run & Download { display-mode: "form" }
import subprocess, pandas as pd, base64
from IPython.display import display, HTML

# Sample output for reference
_output_sample = "identifier_query,identifier_db\nprot_001,NP_123456.1\nprot_002,NP_789012.1\n"
_b64_sample = base64.b64encode(_output_sample.encode()).decode()
display(HTML(f"""
<div style='font-family:sans-serif;margin-bottom:1rem;display:flex;align-items:center;gap:0.75rem;'>
  <span style='font-size:15px;color:#444;'>Example output format:</span>
  <a href='data:text/csv;base64,{_b64_sample}' download='sample_output.csv'
     style='display:inline-block;padding:0.35rem 0.85rem;background:#f0f4ff;color:#1a56db;border:1.5px solid #1a56db;border-radius:4px;text-decoration:none;font-family:sans-serif;font-size:14px;'>
    &#8659; Download sample_output.csv
  </a>
</div>
"""))

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0f4ff;border-left:4px solid #1a56db;border-radius:0 8px 8px 0;font-family:sans-serif;margin-bottom:0.8rem;'>
  <b style='color:#1a56db;'>&#8635; Processing&hellip; this may take a few minutes.</b>
</div>
"""))

species_name = query_filename.split('_')[0]
db_name    = f'{species_name}_db'
blast_out  = f'{species_name}_blast.txt'
output_csv = f'{species_name}_protein_id_mapping.csv'

subprocess.run(['makeblastdb', '-in', db_filename, '-dbtype', 'prot', '-out', db_name],
               check=True, capture_output=True)
subprocess.run([
    'blastp', '-query', query_filename, '-db', db_name, '-out', blast_out,
    '-outfmt', '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore qcovs',
    '-num_threads', '2'
], check=True, capture_output=True)

cols = ['qseqid','sseqid','pident','length','mismatch','gapopen','qstart','qend','sstart','send','evalue','bitscore','qcovs']
blast_df = pd.read_csv(blast_out, sep='\t', header=None, names=cols)

def seq_lengths(fasta):
    lengths, cur_id, cur_len = {}, None, 0
    with open(fasta) as f:
        for line in f:
            if line.startswith('>'):
                if cur_id: lengths[cur_id] = cur_len
                cur_id, cur_len = line[1:].strip().split()[0], 0
            else:
                cur_len += len(line.strip())
    if cur_id: lengths[cur_id] = cur_len
    return lengths

blast_df['qlen'] = blast_df['qseqid'].map(seq_lengths(query_filename))
blast_df['slen'] = blast_df['sseqid'].map(seq_lengths(db_filename))
blast_df['qcov'] = (blast_df['qend'] - blast_df['qstart'] + 1) / blast_df['qlen']
blast_df['scov'] = blast_df['length'] / blast_df['slen']

filtered = blast_df[
    (blast_df['pident']   >= identity)   &
    (blast_df['gapopen']  <= gaps)       &
    (blast_df['mismatch'] <= mismatches) &
    (blast_df['qcov']     >= query_cov   / 100) &
    (blast_df['scov']     >= subject_cov / 100)
]

best   = filtered.sort_values('evalue').groupby('qseqid').first().reset_index()
result = best[['qseqid','sseqid']].rename(columns={'qseqid':'identifier_query','sseqid':'identifier_db'})
result.to_csv(output_csv, index=False)

with open(output_csv, 'rb') as f:
    _csv_b64 = base64.b64encode(f.read()).decode()

display(HTML(f"""
<div style='padding:1rem 1.2rem;background:#f0fff8;border:1.5px solid #006d5b;border-radius:8px;font-family:sans-serif;margin-top:0.5rem;'>
  <b style='font-size:16px;color:#006d5b;'>&#10003; Done! {len(result)} matches found out of {blast_df['qseqid'].nunique()} sequences.</b>
  <div style='margin-top:0.6rem;'>
    <a href='data:text/csv;base64,{_csv_b64}' download='{output_csv}'
       style='display:inline-block;padding:0.5rem 1.4rem;background:#006d5b;color:#fff;border-radius:4px;text-decoration:none;font-family:sans-serif;font-size:15px;font-weight:600;'>
      &#8659; Download {output_csv}
    </a>
  </div>
</div>
"""))

In [ ]:
#@title Protein Sequences Mapper { display-mode: "form" }
from IPython.display import display, HTML
display(HTML("""
<div style='font-family:sans-serif;'>

  <div style='padding:0.7rem 1rem;background:#f8f9fb;border:1px solid #e0e0e0;border-radius:6px;margin-bottom:1.5rem;font-size:13px;color:#666;line-height:1.7;'>
    Copyright &copy; 2026. All rights reserved.<br>
    Free for academic research and education purposes.
    For commercial use or licensing enquiries, please contact
    <a href='mailto:yeohc@a-star.edu.sg' style='color:#1a56db;'>yeohc@a-star.edu.sg</a> or
    <a href='mailto:yeodynasty@yahoo.com.sg' style='color:#1a56db;'>yeodynasty@yahoo.com.sg</a>.<br>
    Refer to the LICENSE file in the root directory for full terms and conditions.
  </div>

  <div style='display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:1.5rem;'>
    <div style='flex:1;'>
      <h1 style='margin:0 0 0.5rem 0;font-size:2rem;font-weight:700;'>Protein Sequences Mapper</h1>
      <p style='margin:0;color:#444;font-size:16px;'>Maps protein sequences from a Genome-scale metabolic model (GEM) to the coding sequences of a reference genome using BLASTp.</p>
    </div>
    <div style='margin-left:2rem;flex-shrink:0;'>
      <img src='https://raw.githubusercontent.com/MeMoModelling/proteinSeqMapping/main/bii_horizontal_logo_smalle472014210204e8886d5cf09c653e7e5.png' style='width:190px;'/>
    </div>
  </div>

  <hr style='border:none;border-top:1px solid #ddd;margin:1.5rem 0;'/>

  <h2 style='font-size:1.4rem;margin:0 0 0.8rem 0;'>Instructions:</h2>
  <ol style='margin:0;padding-left:1.5rem;font-size:16px;line-height:1.8;'>
    <li>Adjust BLASTp parameters in Step 1 (<em>defaults recommended</em>).</li>
    <li>Go to <em>Runtime &rarr; Run all</em> in the top menu.</li>
    <li>Upload your files when prompted in Step 2.</li>
    <li>Download the output CSV from Step 3.</li>
  </ol>

</div>
"""))

In [ ]:
#@title Step 1 — Set up environment { display-mode: "form" }
import shutil, subprocess
from IPython.display import display, HTML

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0f4ff;border-left:4px solid #1a56db;border-radius:0 8px 8px 0;font-family:sans-serif;'>
  <b style='color:#1a56db;'>&#9881; Setting up environment...</b>
  <p style='margin:0.3rem 0 0 0;font-size:0.88rem;color:#555;'>This runs automatically and takes about 1 minute.</p>
</div>
"""))

if not shutil.which('blastp'):
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ncbi-blast+'], check=True, capture_output=True)
subprocess.run(['pip', 'install', '-q', 'pandas'], capture_output=True)

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0fff8;border-left:4px solid #006d5b;border-radius:0 8px 8px 0;font-family:sans-serif;'>
  <b style='color:#006d5b;'>&#10003; Ready! Proceed to the next step.</b>
</div>
"""))

In [ ]:
#@title Step 1 — Adjust parameters { display-mode: "form" }
#@markdown Default values work for most cases.
#@markdown ---
#@markdown **Min % similarity between sequences**
identity    = 95  #@param {type:"slider", min:50, max:100, step:1}
#@markdown **Min % of query sequence that must align**
query_cov   = 85  #@param {type:"slider", min:50, max:100, step:1}
#@markdown **Min % of subject sequence that must align**
subject_cov = 85  #@param {type:"slider", min:50, max:100, step:1}
#@markdown **Max residue differences allowed**
mismatches  = 1   #@param {type:"slider", min:0, max:10, step:1}
#@markdown **Gaps in alignment (0 = none)**
gaps        = 0   #@param {type:"slider", min:0, max:5,  step:1}

In [ ]:
#@title Step 2 — Upload files { display-mode: "form" }
from IPython.display import display, HTML
from google.colab import files
import os, base64

_query_sample = """>prot_001 hypothetical protein
MKVLKFGATLAQTIQTVAETFTLDVYGDHEQRIAQQAKKLLEDIPVQLQ
>prot_002 hypothetical protein
MALKFGATLAQTIQTVAETFTLDVYGDHEQRIAQQAKKLLEDIPVQLQD
"""
_db_sample = """>NP_123456.1 hypothetical protein
MKVLKFGATLAQTIQTVAETFTLDVYGDHEQRIAQQAKKLLEDIPVQLQ
>NP_789012.1 hypothetical protein
MALKFGATLAQTIQTVAETFTLDVYGDHEQRIAQQAKKLLEDIPVQLQD
>NP_345678.1 other protein
MSATGKVIKCKAAVLWEEKKPFSIEEVEVAPPKAHEVRIKMVATGICRSD
"""

def _sample_btn(content, filename):
    b64 = base64.b64encode(content.encode()).decode()
    return (f"<a href='data:text/plain;base64,{b64}' download='{filename}' "
            f"style='display:inline-block;padding:0.35rem 0.85rem;background:#f0f4ff;"
            f"color:#1a56db;border:1.5px solid #1a56db;border-radius:4px;text-decoration:none;"
            f"font-family:sans-serif;font-size:14px;margin-top:0.4rem;'>&#8659; Download {filename}</a>")

display(HTML(f"""
<div style='padding:0.8rem 1.2rem;background:#f8f9fb;border:1.5px solid #d0d7e8;border-radius:8px;font-family:sans-serif;margin-bottom:0.8rem;'>
  <b style='font-size:16px;'>&#128193; Model protein sequences</b>
  <p style='margin:0.2rem 0 0.4rem 0;font-size:15px;color:#555;'>The protein FASTA file from your metabolic model <code>(.faa)</code></p>
  {_sample_btn(_query_sample, 'sample_query.faa')}
</div>
"""))
uploaded_query = files.upload()
query_filename_raw = list(uploaded_query.keys())[0]
query_filename = query_filename_raw.replace(' ', '_')
if query_filename != query_filename_raw:
    os.rename(query_filename_raw, query_filename)
display(HTML(f"<p style='font-family:sans-serif;font-size:15px;color:#006d5b;'>&#10003; Uploaded: <code>{query_filename}</code></p>"))

display(HTML(f"""
<div style='padding:0.8rem 1.2rem;background:#f8f9fb;border:1.5px solid #d0d7e8;border-radius:8px;font-family:sans-serif;margin-bottom:0.8rem;'>
  <b style='font-size:16px;'>&#128193; Genome protein sequences</b>
  <p style='margin:0.2rem 0 0.4rem 0;font-size:15px;color:#555;'>The protein FASTA file as the sequence database <code>(.faa)</code></p>
  {_sample_btn(_db_sample, 'sample_database.faa')}
</div>
"""))
uploaded_db = files.upload()
db_filename_raw = list(uploaded_db.keys())[0]
db_filename = db_filename_raw.replace(' ', '_')
if db_filename != db_filename_raw:
    os.rename(db_filename_raw, db_filename)
display(HTML(f"<p style='font-family:sans-serif;font-size:15px;color:#006d5b;'>&#10003; Uploaded: <code>{db_filename}</code></p>"))